In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline("text2text-generation", model="google/flan-t5-base")
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{instruction}:\n\n{document}")
])


In [ ]:
from langchain.tools import tool

@tool
def fetch_document(file_path: str) -> str:
    """Fetches the content of a local text file."""
    with open("Move_like_water.pdf", "r", encoding="utf-8") as f:
        return f.read()
        
@tool("summary")
def summarize_document(document: str) -> str:
    """Summarize a given document based on the query."""

    prompt_value = prompt.format_prompt(
        instruction="Summarize",
        document=document
    )
    
    text = prompt_value.to_string()
    response = pipe(text)
    return response[0]["generated_text"]
    
    

@tool("analyze")
def analyze_document(document: str) -> str:
    """Analyzes the themes, tone, and key ideas of the given document."""
    
    prompt_value = prompt.format_prompt(
        instruction="Analyze the themes, tone, and key ideas in",
        document=document
    )
    
    text = prompt_value.to_string()
    response = pipe(text)
    return response[0]["generated_text"]
    


In [ ]:
from langchain.agents import initialize_agent, AgentType

tools = [fetch_document, summarize_document, analyze_document]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True
)

In [ ]:
# Example 1: Summarize a local file
agent.run("Summarize the document move_like_water.pdf")

# Example 2: Analyze a local file
agent.run("Analyze the themes of the document move_like_water.pdf")